# 0. Environment Setup

## 0.1 Install Dependencies

In [78]:
!pip install ultralytics -q

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\xlosn\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\matplotlib\\backends\\_backend_agg.cp311-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



## 0.2 Import Libraries

In [79]:
import os
import json
import shutil
from pathlib import Path
from collections import defaultdict, Counter

import yaml
import torch
import ultralytics

from ultralytics import YOLO
ultralytics.checks()

Ultralytics 8.4.144  Python-3.11.9 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce MX450, 2048MiB)
Setup complete  (8 CPUs, 15.8 GB RAM, 507.3/952.5 GB disk)


# 1. Dataset Configuration

## 1.1 Define Dataset Paths

In [80]:
DATASET_DIR = Path(
    r"C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_COCO"
)

ANNOTATIONS_DIR = DATASET_DIR / "annotations"

TRAIN_JSON = ANNOTATIONS_DIR / "instances_train2017.json"
VAL_JSON   = ANNOTATIONS_DIR / "instances_val2017.json"
TEST_JSON  = ANNOTATIONS_DIR / "instances_test2017.json"

TRAIN_IMAGES = DATASET_DIR / "train2017"
VAL_IMAGES   = DATASET_DIR / "val2017"
TEST_IMAGES  = DATASET_DIR / "test2017"

print("Dataset:", DATASET_DIR)
print("Annotations:", ANNOTATIONS_DIR)

print("\nJSON files:")
print(TRAIN_JSON)
print(VAL_JSON)
print(TEST_JSON)

print("\nImages:")
print("Train:", TRAIN_IMAGES.exists())
print("Val:", VAL_IMAGES.exists())
print("Test:", TEST_IMAGES.exists())

Dataset: C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_COCO
Annotations: C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_COCO\annotations

JSON files:
C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_COCO\annotations\instances_train2017.json
C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_COCO\annotations\instances_val2017.json
C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_COCO\annotations\instances_test2017.json

Images:
Train: True
Val: True
Test: True


# 2. Dataset Overview

# 2.1 Dataset Statistics

In [81]:
for name, json_file in [
    ("Train", TRAIN_JSON),
    ("Validation", VAL_JSON),
    ("Test", TEST_JSON)
]:
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"\n{name}")
    print("Images:", len(data["images"]))
    print("Annotations:", len(data["annotations"]))
    print("Categories:", len(data["categories"]))
    print("Classes:", [c["name"] for c in data["categories"]])


Train
Images: 2816
Annotations: 6211
Categories: 6
Classes: ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']

Validation
Images: 810
Annotations: 1744
Categories: 6
Classes: ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']

Test
Images: 374
Annotations: 785
Categories: 6
Classes: ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']


In [82]:
import random
import cv2
import matplotlib

ROOT = Path(r"C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_YOLO")

IMAGE_DIR = ROOT / "images" / "train"
LABEL_DIR = ROOT / "labels" / "train"

class_names = {
    0: "dent",
    1: "scratch",
    2: "crack",
    3: "glass shatter",
    4: "lamp broken",
    5: "tire flat",
}

# Collect images for each class
images_by_class = {i: [] for i in class_names}

for label_file in LABEL_DIR.glob("*.txt"):

    lines = label_file.read_text(encoding="utf-8").splitlines()

    classes_in_image = set()

    for line in lines:
        parts = line.split()

        if len(parts) == 5:
            cls = int(float(parts[0]))
            classes_in_image.add(cls)

    for cls in classes_in_image:
        if cls in images_by_class:
            image_file = IMAGE_DIR / (label_file.stem + ".jpg")

            if image_file.exists():
                images_by_class[cls].append(image_file)


# Show 3 random examples from each class
random.seed(42)

for cls, name in class_names.items():

    samples = random.sample(
        images_by_class[cls],
        min(3, len(images_by_class[cls]))
    )

    print(f"\n{name}: {len(images_by_class[cls])} images")

    for image_file in samples:

        image = cv2.imread(str(image_file))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        label_file = LABEL_DIR / (image_file.stem + ".txt")

        h, w = image.shape[:2]

        # Draw ALL annotations in this image
        for line in label_file.read_text(encoding="utf-8").splitlines():

            parts = line.split()

            if len(parts) != 5:
                continue

            c, xc, yc, bw, bh = map(float, parts)

            c = int(c)

            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)

            cv2.rectangle(
                image,
                (x1, y1),
                (x2, y2),
                (255, 0, 0),
                2
            )

            cv2.putText(
                image,
                class_names[c],
                (x1, max(y1 - 5, 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 0, 0),
                2
            )

        import matplotlib.pyplot as plt

        matplotlib.use("Agg", force=True)
        import matplotlib.pyplot as plt
        import matplotlib.pyplot as plt
        plt.imshow(image)
        plt.title(f"{name} | {image_file.name}")
        plt.axis("off")
        plt.show()


dent: 1242 images

scratch: 1507 images

crack: 434 images


C:\Users\xlosn\AppData\Local\Temp\ipykernel_11688\3524138528.py:107: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



glass shatter: 469 images

lamp broken: 489 images

tire flat: 219 images


In [83]:
CLASS_ID = 0
CLASS_NAME = "dent"

dent_images = []

for label_file in LABEL_DIR.glob("*.txt"):

    for line in label_file.read_text(encoding="utf-8").splitlines():

        parts = line.split()

        if len(parts) == 5 and int(float(parts[0])) == CLASS_ID:
            image_file = IMAGE_DIR / f"{label_file.stem}.jpg"

            if image_file.exists():
                dent_images.append(image_file)

            break

print("Total images containing dent:", len(dent_images))

random.seed(789)

samples = random.sample(
    dent_images,
    min(10, len(dent_images))
)

for image_file in samples:

    image = cv2.imread(str(image_file))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    label_file = LABEL_DIR / f"{image_file.stem}.txt"

    h, w = image.shape[:2]

    for line in label_file.read_text(encoding="utf-8").splitlines():

        parts = line.split()

        if len(parts) != 5:
            continue

        c, xc, yc, bw, bh = map(float, parts)
        c = int(c)

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            2
        )

        cv2.putText(
            image,
            class_names[c],
            (x1, max(y1 - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )

    plt.figure(figsize=(9, 7))
    plt.imshow(image)
    plt.title(f"DENT | {image_file.name}")
    plt.axis("off")
    plt.show()

Total images containing dent: 1242


C:\Users\xlosn\AppData\Local\Temp\ipykernel_11688\2471656990.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [84]:
CLASS_ID = 1
CLASS_NAME = "scratch"

scratch_images = []

for label_file in LABEL_DIR.glob("*.txt"):

    for line in label_file.read_text(encoding="utf-8").splitlines():

        parts = line.split()

        if len(parts) == 5 and int(float(parts[0])) == CLASS_ID:
            image_file = IMAGE_DIR / f"{label_file.stem}.jpg"

            if image_file.exists():
                scratch_images.append(image_file)

            break

print("Total images containing scratch:", len(scratch_images))

random.seed(456)

samples = random.sample(
    scratch_images,
    min(10, len(scratch_images))
)

for image_file in samples:

    image = cv2.imread(str(image_file))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    label_file = LABEL_DIR / f"{image_file.stem}.txt"

    h, w = image.shape[:2]

    for line in label_file.read_text(encoding="utf-8").splitlines():

        parts = line.split()

        if len(parts) != 5:
            continue

        c, xc, yc, bw, bh = map(float, parts)
        c = int(c)

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            2
        )

        cv2.putText(
            image,
            class_names[c],
            (x1, max(y1 - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )

    plt.figure(figsize=(9, 7))
    plt.imshow(image)
    plt.title(f"SCRATCH | {image_file.name}")
    plt.axis("off")
    plt.show()

Total images containing scratch: 1507


C:\Users\xlosn\AppData\Local\Temp\ipykernel_11688\3095321173.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [85]:
CLASS_ID = 3
CLASS_NAME = "glass shatter"

glass_images = []

for label_file in LABEL_DIR.glob("*.txt"):

    for line in label_file.read_text(encoding="utf-8").splitlines():

        parts = line.split()

        if len(parts) == 5 and int(float(parts[0])) == CLASS_ID:
            image_file = IMAGE_DIR / f"{label_file.stem}.jpg"

            if image_file.exists():
                glass_images.append(image_file)

            break

print("Total images containing glass shatter:", len(glass_images))

random.seed(321)

samples = random.sample(
    glass_images,
    min(10, len(glass_images))
)

for image_file in samples:

    image = cv2.imread(str(image_file))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    label_file = LABEL_DIR / f"{image_file.stem}.txt"

    h, w = image.shape[:2]

    for line in label_file.read_text(encoding="utf-8").splitlines():

        parts = line.split()

        if len(parts) != 5:
            continue

        c, xc, yc, bw, bh = map(float, parts)
        c = int(c)

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            2
        )

        cv2.putText(
            image,
            class_names[c],
            (x1, max(y1 - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )

    plt.figure(figsize=(9, 7))
    plt.imshow(image)
    plt.title(f"GLASS SHATTER | {image_file.name}")
    plt.axis("off")
    plt.show()

Total images containing glass shatter: 469


C:\Users\xlosn\AppData\Local\Temp\ipykernel_11688\187132350.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [86]:
CLASS_ID = 4
CLASS_NAME = "lamp broken"

lamp_images = []

for label_file in LABEL_DIR.glob("*.txt"):

    for line in label_file.read_text(encoding="utf-8").splitlines():

        parts = line.split()

        if len(parts) == 5 and int(float(parts[0])) == CLASS_ID:
            image_file = IMAGE_DIR / f"{label_file.stem}.jpg"

            if image_file.exists():
                lamp_images.append(image_file)

            break

print("Total images containing lamp broken:", len(lamp_images))

random.seed(654)

samples = random.sample(
    lamp_images,
    min(10, len(lamp_images))
)

for image_file in samples:

    image = cv2.imread(str(image_file))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    label_file = LABEL_DIR / f"{image_file.stem}.txt"

    h, w = image.shape[:2]

    for line in label_file.read_text(encoding="utf-8").splitlines():

        parts = line.split()

        if len(parts) != 5:
            continue

        c, xc, yc, bw, bh = map(float, parts)
        c = int(c)

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            2
        )

        cv2.putText(
            image,
            class_names[c],
            (x1, max(y1 - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )

    plt.figure(figsize=(9, 7))
    plt.imshow(image)
    plt.title(f"LAMP BROKEN | {image_file.name}")
    plt.axis("off")
    plt.show()

Total images containing lamp broken: 489


C:\Users\xlosn\AppData\Local\Temp\ipykernel_11688\4178080234.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [87]:
CLASS_ID = 5
CLASS_NAME = "tire flat"

tire_images = []

for label_file in LABEL_DIR.glob("*.txt"):

    for line in label_file.read_text(encoding="utf-8").splitlines():

        parts = line.split()

        if len(parts) == 5 and int(float(parts[0])) == CLASS_ID:
            image_file = IMAGE_DIR / f"{label_file.stem}.jpg"

            if image_file.exists():
                tire_images.append(image_file)

            break

print("Total images containing tire flat:", len(tire_images))

random.seed(987)

samples = random.sample(
    tire_images,
    min(10, len(tire_images))
)

for image_file in samples:

    image = cv2.imread(str(image_file))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    label_file = LABEL_DIR / f"{image_file.stem}.txt"

    h, w = image.shape[:2]

    for line in label_file.read_text(encoding="utf-8").splitlines():

        parts = line.split()

        if len(parts) != 5:
            continue

        c, xc, yc, bw, bh = map(float, parts)
        c = int(c)

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            2
        )

        cv2.putText(
            image,
            class_names[c],
            (x1, max(y1 - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )

    plt.figure(figsize=(9, 7))
    plt.imshow(image)
    plt.title(f"TIRE FLAT | {image_file.name}")
    plt.axis("off")
    plt.show()

Total images containing tire flat: 219


C:\Users\xlosn\AppData\Local\Temp\ipykernel_11688\1645067283.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [76]:
from pathlib import Path
from collections import defaultdict
import numpy as np

ROOT = Path(
    r"C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_YOLO"
)

IMAGE_DIR = ROOT / "images" / "train"
LABEL_DIR = ROOT / "labels" / "train"

class_names = {
    0: "dent",
    1: "scratch",
    2: "crack",
    3: "glass shatter",
    4: "lamp broken",
    5: "tire flat",
}

# Store normalized bounding-box areas
areas = defaultdict(list)

for label_file in LABEL_DIR.glob("*.txt"):

    for line in label_file.read_text(encoding="utf-8").splitlines():

        parts = line.split()

        if len(parts) != 5:
            continue

        cls, xc, yc, w, h = map(float, parts)
        cls = int(cls)

        if cls in class_names:
            # YOLO normalized bbox area
            area = w * h
            areas[cls].append(area)

print("=" * 65)
print("BOUNDING BOX SIZE ANALYSIS - TRAIN")
print("=" * 65)

for cls in class_names:

    values = np.array(areas[cls])

    print(f"\n{class_names[cls]}")
    print("-" * 30)

    print("Annotations :", len(values))
    print("Minimum     :", round(values.min(), 6))
    print("25th %      :", round(np.percentile(values, 25), 6))
    print("Median      :", round(np.median(values), 6))
    print("Mean        :", round(values.mean(), 6))
    print("75th %      :", round(np.percentile(values, 75), 6))
    print("Maximum     :", round(values.max(), 6))

print("\n" + "=" * 65)

BOUNDING BOX SIZE ANALYSIS - TRAIN

dent
------------------------------
Annotations : 1806
Minimum     : 0.000202
25th %      : 0.032337
Median      : 0.082557
Mean        : 0.135896
75th %      : 0.185439
Maximum     : 1.0

scratch
------------------------------
Annotations : 2560
Minimum     : 0.000849
25th %      : 0.019063
Median      : 0.054203
Mean        : 0.11239
75th %      : 0.147564
Maximum     : 1.0

crack
------------------------------
Annotations : 651
Minimum     : 0.000175
25th %      : 0.003619
Median      : 0.009688
Mean        : 0.031568
75th %      : 0.027671
Maximum     : 0.777377

glass shatter
------------------------------
Annotations : 475
Minimum     : 0.014845
25th %      : 0.441176
Median      : 0.667816
Mean        : 0.641034
75th %      : 0.882732
Maximum     : 1.0

lamp broken
------------------------------
Annotations : 494
Minimum     : 0.002513
25th %      : 0.091239
Median      : 0.185859
Mean        : 0.240267
75th %      : 0.352199
Maximum     : 1.0

# 2.2 Class Distribution

In [71]:
for split, ann_file in {
    "train": "instances_train2017.json",
    "val": "instances_val2017.json",
    "test": "instances_test2017.json"
}.items():

    coco = json.loads(
        (ANNOTATIONS_DIR / ann_file).read_text(encoding="utf-8")
    )

    print(f"\n{'='*50}")
    print(split.upper())
    print(f"{'='*50}")

    print("\nCategories:")
    for cat in coco["categories"]:
        print(f"ID: {cat['id']} | Name: {cat['name']}")

    counts = Counter(
        ann["category_id"]
        for ann in coco["annotations"]
    )

    print("\nAnnotation counts by category ID:")

    for cat_id, count in sorted(counts.items()):
        name = next(
            (
                cat["name"]
                for cat in coco["categories"]
                if cat["id"] == cat_id
            ),
            "UNKNOWN"
        )

        print(f"ID {cat_id}: {count} annotations -> {name}")


TRAIN

Categories:
ID: 1 | Name: dent
ID: 2 | Name: scratch
ID: 3 | Name: crack
ID: 4 | Name: glass shatter
ID: 5 | Name: lamp broken
ID: 6 | Name: tire flat

Annotation counts by category ID:
ID 1: 1806 annotations -> dent
ID 2: 2560 annotations -> scratch
ID 3: 651 annotations -> crack
ID 4: 475 annotations -> glass shatter
ID 5: 494 annotations -> lamp broken
ID 6: 225 annotations -> tire flat

VAL

Categories:
ID: 1 | Name: dent
ID: 2 | Name: scratch
ID: 3 | Name: crack
ID: 4 | Name: glass shatter
ID: 5 | Name: lamp broken
ID: 6 | Name: tire flat

Annotation counts by category ID:
ID 1: 501 annotations -> dent
ID 2: 728 annotations -> scratch
ID 3: 177 annotations -> crack
ID 4: 135 annotations -> glass shatter
ID 5: 141 annotations -> lamp broken
ID 6: 62 annotations -> tire flat

TEST

Categories:
ID: 1 | Name: dent
ID: 2 | Name: scratch
ID: 3 | Name: crack
ID: 4 | Name: glass shatter
ID: 5 | Name: lamp broken
ID: 6 | Name: tire flat

Annotation counts by category ID:
ID 1: 236 

# 3. Data Preparation

## 3.1 Create YOLO Directory Structure

In [72]:
YOLO_DIR = DATASET_DIR.parent / "CarDD_YOLO"

for split in ["train", "val", "test"]:
    (YOLO_DIR / "images" / split).mkdir(
        parents=True,
        exist_ok=True
    )

    (YOLO_DIR / "labels" / split).mkdir(
        parents=True,
        exist_ok=True
    )

print("YOLO dataset directory:")
print(YOLO_DIR)

YOLO dataset directory:
C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_YOLO


## 3.2 Convert COCO Annotations to YOLO Format

In [73]:
def convert_coco_to_yolo(
    json_file,
    images_dir,
    output_images_dir,
    output_labels_dir
):
    with open(json_file, "r", encoding="utf-8") as f:
        coco = json.load(f)

    # Sort categories by COCO category ID
    categories = sorted(
        coco["categories"],
        key=lambda x: x["id"]
    )

    # COCO category ID -> YOLO class ID
    category_mapping = {
        cat["id"]: i
        for i, cat in enumerate(categories)
    }

    # Image ID -> image information
    images = {
        img["id"]: img
        for img in coco["images"]
    }

    # Group annotations by image
    annotations_by_image = {}

    for ann in coco["annotations"]:
        image_id = ann["image_id"]

        if image_id not in annotations_by_image:
            annotations_by_image[image_id] = []

        annotations_by_image[image_id].append(ann)

    # Convert each image
    for image_id, image_info in images.items():

        file_name = image_info["file_name"]
        width = image_info["width"]
        height = image_info["height"]

        source_image = images_dir / file_name
        destination_image = (
            output_images_dir / Path(file_name).name
        )

        # Copy image
        if source_image.exists():
            shutil.copy2(
                source_image,
                destination_image
            )
        else:
            print(
                f"WARNING: Image not found: {source_image}"
            )
            continue

        # Create YOLO label file
        label_file = (
            output_labels_dir
            / (Path(file_name).stem + ".txt")
        )

        with open(
            label_file,
            "w",
            encoding="utf-8"
        ) as f:

            for ann in annotations_by_image.get(
                image_id, []
            ):

                x, y, w, h = ann["bbox"]

                # COCO bbox -> YOLO normalized format
                x_center = (x + w / 2) / width
                y_center = (y + h / 2) / height

                w_norm = w / width
                h_norm = h / height

                class_id = category_mapping[
                    ann["category_id"]
                ]

                f.write(
                    f"{class_id} "
                    f"{x_center:.6f} "
                    f"{y_center:.6f} "
                    f"{w_norm:.6f} "
                    f"{h_norm:.6f}\n"
                )

    return [
        cat["name"]
        for cat in categories
    ]

## 3.3 Convert All Dataset Splits

In [74]:
train_classes = convert_coco_to_yolo(
    TRAIN_JSON,
    TRAIN_IMAGES,
    YOLO_DIR / "images" / "train",
    YOLO_DIR / "labels" / "train"
)

val_classes = convert_coco_to_yolo(
    VAL_JSON,
    VAL_IMAGES,
    YOLO_DIR / "images" / "val",
    YOLO_DIR / "labels" / "val"
)

test_classes = convert_coco_to_yolo(
    TEST_JSON,
    TEST_IMAGES,
    YOLO_DIR / "images" / "test",
    YOLO_DIR / "labels" / "test"
)

print("Conversion complete!")
print("Classes:", train_classes)

KeyboardInterrupt: 

# 4. Dataset Validation

## 4.1 Verify Image-Label Matching

In [ ]:
for split in ["train", "val", "test"]:

    images_dir = YOLO_DIR / "images" / split
    labels_dir = YOLO_DIR / "labels" / split

    images = list(images_dir.glob("*.jpg"))
    labels = list(labels_dir.glob("*.txt"))

    print(f"\n{split.upper()}")
    print(f"Images : {len(images)}")
    print(f"Labels : {len(labels)}")

    image_stems = {
        img.stem for img in images
    }

    label_stems = {
        lbl.stem for lbl in labels
    }

    missing_labels = image_stems - label_stems
    extra_labels = label_stems - image_stems

    print(
        f"Images without labels : "
        f"{len(missing_labels)}"
    )

    print(
        f"Labels without images : "
        f"{len(extra_labels)}"
    )


TRAIN
Images : 2816
Labels : 2816
Images without labels : 0
Labels without images : 0

VAL
Images : 810
Labels : 810
Images without labels : 0
Labels without images : 0

TEST
Images : 374
Labels : 374
Images without labels : 0
Labels without images : 0


## 4.2 Validate YOLO Label Format

In [ ]:
# Check YOLO label format and class IDs

class_names = [
    "dent",
    "scratch",
    "crack",
    "glass shatter",
    "lamp broken",
    "tire flat"
]

for split in ["train", "val", "test"]:
    labels_dir = YOLO_DIR / "labels" / split

    invalid_lines = []
    class_counts = {i: 0 for i in range(len(class_names))}
    total_boxes = 0

    for label_file in labels_dir.glob("*.txt"):

        with open(label_file, "r", encoding="utf-8") as f:
            lines = f.readlines()

        for line_number, line in enumerate(lines, start=1):

            parts = line.strip().split()

            # YOLO format must contain 5 values
            if len(parts) != 5:
                invalid_lines.append(
                    f"{label_file.name} - line {line_number}"
                )
                continue

            class_id, x, y, w, h = map(float, parts)

            class_id = int(class_id)

            # Check class ID
            if class_id not in class_counts:
                invalid_lines.append(
                    f"{label_file.name} - invalid class {class_id}"
                )
                continue

            # Check normalized coordinates
            if not (
                0 <= x <= 1 and
                0 <= y <= 1 and
                0 < w <= 1 and
                0 < h <= 1
            ):
                invalid_lines.append(
                    f"{label_file.name} - invalid bbox"
                )
                continue

            class_counts[class_id] += 1
            total_boxes += 1

    print(f"\n{'='*40}")
    print(f"{split.upper()}")
    print(f"{'='*40}")

    print("Total bounding boxes:", total_boxes)

    print("\nClass distribution:")
    for class_id, count in class_counts.items():
        print(f"{class_id}: {class_names[class_id]:15s} -> {count}")

    print("\nInvalid label entries:", len(invalid_lines))

    if invalid_lines:
        print("Examples:")
        for item in invalid_lines[:10]:
            print(" ", item)


TRAIN
Total bounding boxes: 6211

Class distribution:
0: dent            -> 1806
1: scratch         -> 2560
2: crack           -> 651
3: glass shatter   -> 475
4: lamp broken     -> 494
5: tire flat       -> 225

Invalid label entries: 0

VAL
Total bounding boxes: 1744

Class distribution:
0: dent            -> 501
1: scratch         -> 728
2: crack           -> 177
3: glass shatter   -> 135
4: lamp broken     -> 141
5: tire flat       -> 62

Invalid label entries: 0

TEST
Total bounding boxes: 785

Class distribution:
0: dent            -> 236
1: scratch         -> 307
2: crack           -> 70
3: glass shatter   -> 71
4: lamp broken     -> 69
5: tire flat       -> 32

Invalid label entries: 0


## 4.3 Validate COCO-to-YOLO Conversion

In [ ]:
ROOT = Path(r"C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release")
COCO_ROOT = ROOT / "CarDD_COCO"
YOLO_ROOT = ROOT / "CarDD_YOLO"

splits = {
    "train": ("instances_train2017.json", "train"),
    "val": ("instances_val2017.json", "val"),
    "test": ("instances_test2017.json", "test"),
}

class_names = {
    0: "dent",
    1: "scratch",
    2: "crack",
    3: "glass shatter",
    4: "lamp broken",
    5: "tire flat",
}

for split, (ann_file, yolo_split) in splits.items():

    coco_file = COCO_ROOT / "annotations" / ann_file
    coco = json.loads(coco_file.read_text(encoding="utf-8"))

    images = {im["id"]: im for im in coco["images"]}
    anns = coco["annotations"]

    missing_images = []
    bad_class = []
    bad_bbox = []
    bad_area = []

    # Check every COCO annotation
    for a in anns:

        img = images.get(a["image_id"])

        if img is None:
            missing_images.append(a["id"])
            continue

        # Class check
        cls = a["category_id"]

        if cls not in class_names:
            bad_class.append((a["id"], cls))

        # Bounding box check
        x, y, w, h = a["bbox"]
        W, H = img["width"], img["height"]

        if w <= 0 or h <= 0:
            bad_bbox.append(
                (a["id"], "non-positive", a["bbox"])
            )

        if x < 0 or y < 0 or x + w > W or y + h > H:
            bad_bbox.append(
                (a["id"], "out-of-bounds", a["bbox"], (W, H))
            )

        # Area check
        if a.get("area", 1) <= 0:
            bad_area.append(
                (a["id"], a.get("area"))
            )

    # Check YOLO labels
    label_dir = YOLO_ROOT / "labels" / yolo_split

    yolo_lines = 0
    yolo_files = 0
    yolo_errors = []

    for txt in label_dir.glob("*.txt"):

        yolo_files += 1

        for line_no, line in enumerate(
            txt.read_text(encoding="utf-8").splitlines(), 1
        ):

            if not line.strip():
                continue

            yolo_lines += 1
            parts = line.split()

            # Must have exactly:
            # class x_center y_center width height
            if len(parts) != 5:
                yolo_errors.append(
                    (txt.name, line_no, line)
                )
                continue

            try:
                cls, xc, yc, ww, hh = map(float, parts)

                # Class validation
                if int(cls) != cls or int(cls) not in class_names:
                    yolo_errors.append(
                        (txt.name, line_no, "bad class", line)
                    )

                # Normalized bbox validation
                if not (
                    0 <= xc <= 1
                    and 0 <= yc <= 1
                    and 0 < ww <= 1
                    and 0 < hh <= 1
                ):
                    yolo_errors.append(
                        (txt.name, line_no, "bad normalized bbox", line)
                    )

            except Exception:
                yolo_errors.append(
                    (txt.name, line_no, "not numeric", line)
                )

    print(f"\n{'='*50}")
    print(f"{split.upper()}")
    print(f"{'='*50}")

    print("COCO images:", len(images))
    print("COCO annotations:", len(anns))

    print("YOLO label files:", yolo_files)
    print("YOLO annotation lines:", yolo_lines)

    print("Missing image refs:", len(missing_images))
    print("Bad class IDs:", len(bad_class))
    print("Bad COCO boxes:", len(bad_bbox))
    print("Bad area:", len(bad_area))
    print("Bad YOLO lines:", len(yolo_errors))

    print(
        "COCO ↔ YOLO annotation count match:",
        len(anns) == yolo_lines
    )


TRAIN
COCO images: 2816
COCO annotations: 6211
YOLO label files: 2816
YOLO annotation lines: 6211
Missing image refs: 0
Bad class IDs: 225
Bad COCO boxes: 0
Bad area: 0
Bad YOLO lines: 0
COCO ↔ YOLO annotation count match: True

VAL
COCO images: 810
COCO annotations: 1744
YOLO label files: 810
YOLO annotation lines: 1744
Missing image refs: 0
Bad class IDs: 62
Bad COCO boxes: 0
Bad area: 0
Bad YOLO lines: 0
COCO ↔ YOLO annotation count match: True

TEST
COCO images: 374
COCO annotations: 785
YOLO label files: 374
YOLO annotation lines: 785
Missing image refs: 0
Bad class IDs: 32
Bad COCO boxes: 0
Bad area: 0
Bad YOLO lines: 0
COCO ↔ YOLO annotation count match: True


# 5. Create YOLO Dataset Configuration

In [ ]:
DATASET_DIR = Path(
    r"C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_COCO"
)

YOLO_DIR = DATASET_DIR.parent / "CarDD_YOLO"
DATA_YAML = YOLO_DIR / "data.yaml"

print("YOLO_DIR:", YOLO_DIR)
print("DATA_YAML:", DATA_YAML)
print("Exists:", DATA_YAML.exists())

YOLO_DIR: C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_YOLO
DATA_YAML: C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_YOLO\data.yaml
Exists: True


## 5.1 Create data.yaml

In [ ]:
yaml_content = f"""path: {YOLO_DIR.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: dent
  1: scratch
  2: crack
  3: glass shatter
  4: lamp broken
  5: tire flat
"""

DATA_YAML.write_text(yaml_content, encoding="utf-8")

print("Created:", DATA_YAML)
print(DATA_YAML.read_text(encoding="utf-8"))

Created: C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_YOLO\data.yaml
path: C:/Users/xlosn/qaddir-vehicle-damage-assessment/data/CarDD_release/CarDD_YOLO
train: images/train
val: images/val
test: images/test

names:
  0: dent
  1: scratch
  2: crack
  3: glass shatter
  4: lamp broken
  5: tire flat



# 6. Hardware & Training Environment

Check GPU Availability

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
    print(
        "CUDA version:",
        torch.version.cuda
    )
else:
    print("No NVIDIA CUDA GPU detected.")

CUDA available: True
GPU: NVIDIA GeForce MX450
CUDA version: 12.6


# 7. Baseline Model Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=10,
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    amp=False,
    project="runs/cardd",
    name="baseline_10ep",
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.146 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.144  Python-3.11.9 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce MX450, 2048MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_YOLO\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.0

In [88]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\xlosn\qaddir-vehicle-damage-assessment")

for path in sorted(PROJECT_ROOT.iterdir()):
    print(path.name)

.git
.gitignore
data
docs
notebooks
Qaddir-TaskPlan-Clear-Descriptions.xlsx
README.md
requirements.txt


In [89]:
DOCS_DIR = PROJECT_ROOT / "docs"

for path in sorted(DOCS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))

docs\references.md


In [90]:
references_file = PROJECT_ROOT / "docs" / "references.md"

print(references_file.read_text(encoding="utf-8"))

# References

- [Professional Standards for Vehicle Damage Assessment](https://taqeem.gov.sa/web/content/portal.library/266/attachment_ar?download=false) — Saudi Authority for Accredited Valuers (Taqeem)



In [94]:
import os

for root, dirs, files in os.walk(PROJECT_ROOT):
    # تجاهل مجلدات البيانات وبيئة Git
    dirs[:] = [d for d in dirs if d not in {".git", "__pycache__"} and d != "data"]

    for file in files:
        if file.endswith((".md", ".txt", ".yaml", ".yml", ".ipynb")):
            file_path = Path(root) / file
            try:
                text = file_path.read_text(encoding="utf-8", errors="ignore")
                if "CarDD" in text or "Car Damage Detection Dataset" in text:
                    print(file_path.relative_to(PROJECT_ROOT))
            except Exception:
                pass

notebooks\01_cardd_data_analysis.ipynb
notebooks\02_cardd_annotation_conversion.ipynb
notebooks\07_cardd_baseline_model.ipynb
notebooks\runs\detect\runs\cardd\baseline_10ep\args.yaml
notebooks\runs\detect\runs\cardd\baseline_test\args.yaml
notebooks\runs\detect\runs\cardd\gpu_test\args.yaml


In [95]:
analysis_notebook = PROJECT_ROOT / "notebooks" / "01_cardd_data_analysis.ipynb"

print(analysis_notebook.exists())

True


In [ ]:
import json

analysis_notebook = PROJECT_ROOT / "notebooks" / "01_cardd_data_analysis.ipynb"

with open(analysis_notebook, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "markdown":
        text = "".join(cell["source"]).strip()
        if text:
            print(f"\n--- MARKDOWN CELL {i} ---")
            print(text)

FileNotFoundError: [Errno 2] No such file or directory: 'notebooks\\01_cardd_data_analysis.ipynb'

In [93]:
README_FILE = PROJECT_ROOT / "README.md"

print(README_FILE.read_text(encoding="utf-8"))


# Qaddir: AI-Assisted Preliminary Vehicle Damage Assessment

Qaddir is a graduation project exploring AI-assisted preliminary assessment of visible vehicle damage from images.

The project aims to identify damage types, locate affected vehicle parts, and produce structured results for human review.

## Current Phase

Dataset preparation and preprocessing.

## Data

Datasets are not included in this repository.

## Disclaimer

This project is a proof of concept and does not replace professional vehicle assessment.

